# MOTEL dev2 Workspace Notebook

This notebook provides a basic end-to-end workflow for:

1. [show info] show the guideline and structure about what are the user input to add data to the motel-db
2. [user input] users provide the data to be added to the database
3. [data validation] check if the user input match the format/structure
4. [data mapping suggestion] suggest the link to the linked dataset and control vocabulary
5. [user confirmation] users to comfirm/revise the mapping to the linked dataset and control vocabulary
6. [add to the database] the mapped data are created, if neeed, add new data to linked dataset and control vocabulary

---
## How to use start?

1. use the reFuel.ch data to create the first linked dataset and control vocabulary; but no record yet

add a few testing record to the data using the above workflow.

## Step 1 - Show Info and Input Structure

This cell sets paths and loads the schema so you can see the expected structure before adding data.

In [ ]:
## load required packages and paths
from pathlib import Path
from datetime import date
import csv
import difflib
import re
from pprint import pprint

import yaml

CWD = Path.cwd()
DEV2_DIR = CWD if (CWD / "motel-db").exists() else (CWD / "dev2")
DB_DIR = DEV2_DIR / "motel-db"
SCHEMA_PATH = DEV2_DIR / "schema" / "motel_schema.yaml"
TEST_RECORD_PATH = DEV2_DIR / "data" / "test_records" / "r00001.yaml"

print(f"Working directory: {CWD}")
print(f"Using dev2 directory: {DEV2_DIR}")
print(f"Database directory exists: {DB_DIR.exists()}")
print(f"Schema file exists: {SCHEMA_PATH.exists()}")
print(f"Sample input exists: {TEST_RECORD_PATH.exists()}")

Working directory: e:\Barton\repositories\motel-platform\dev2
Using dev2 directory: e:\Barton\repositories\motel-platform\dev2
Database directory exists: True
Schema file exists: True
Sample input exists: True


In [ ]:
## load the schema
if SCHEMA_PATH.exists():
    with SCHEMA_PATH.open("r", encoding="utf-8") as f:
        schema_dict = yaml.safe_load(f)

        print("Schema section: unmapped_record\n")
        pprint(schema_dict.get("unmapped_record", {}),
            width=100,      # Wider lines = less wrapping
            indent=1,       # 2-space indentation (matches your file)
            compact=True,   # Reduces extra blank lines between elements
            sort_dicts=False # Preserves insertion order
            )
else:
    print(f"Schema file not found: {SCHEMA_PATH}")

Schema section: unmapped_record

{'_type': 'records for a single technology, with all information in free-text format (no foreign '
          'keys to controlled vocabulary)',
 'technology_name': 'Common name of the technology (if known, e.g., "Solar PV", "Lithium-ion '
                    'Battery")',
 'description': 'Free-text description of the technology or process (e.g., "A 5 kW rooftop '
                'photovoltaic system in Switzerland with 18% efficiency")',
 'technology': {'_description': 'Technology attributes (no foreign keys)',
                'technology_type': '(optional) Type of technology (e.g., conversion, storage, '
                                   'distribution, consumption)',
                'technology_category': '(optional) Category of technology (e.g., renewable, '
                                       'non-renewable, synthetic)',
                'technology_assumption': '(optional) Any specific assumptions about the technology '
                            

In [3]:
## template

# Legend:
# [REQUIRED] = Essential field, must contain data
# [OPTIONAL] = Can be None or omitted if not applicable

unmapped_record = {
    # --- CORE IDENTIFICATION ---
    "technology_name": "5 kW Residential Rooftop Solar PV",  # [REQUIRED] Common name
    "description": "A 5 kW rooftop photovoltaic system in Switzerland with 18% efficiency, installed on a south-facing slope.",  # [REQUIRED] Free-text description
    
    # --- TECHNOLOGY ATTRIBUTES ---
    "technology": {
        "technology_type": "conversion",           # [OPTIONAL] e.g., conversion, storage, distribution
        "technology_category": "renewable",        # [OPTIONAL] e.g., renewable, non-renewable
        "technology_assumption": "Data based on 2024 Swiss Federal Office of Energy (SFOE) demo plant specs.",  # [OPTIONAL]
        "process_name": "Electricity Generation",  # [OPTIONAL]
        "process_type": "conversion",              # [OPTIONAL]
        "process_category": "power",               # [OPTIONAL]
        "process_assumption": "Assuming standard irradiance conditions for Zurich region.",  # [OPTIONAL]
    },
    
    # --- SCOPE DEFINITION ---
    "scope": {
        "geographic_scope_description": "Switzerland (Zurich Canton)",  # [REQUIRED]
        "temporal_scope_description": "2025-2030",                      # [REQUIRED]
        "capacity_scope_description": "residential (1-10 kW)",          # [REQUIRED]
        "system_boundary_description": "grid-connected, baseload operation",  # [REQUIRED]
        "scope_assumption": "Data from German benchmarks adjusted for Swiss albedo effects.",  # [OPTIONAL]
    },
    
    # --- SOURCES (List of items) ---
    "sources": {
        "_item": [
            {
                "source_description": "Swiss Federal Office of Energy (SFOE) Annual Report 2024",  # [REQUIRED]
                "source_type": "report",                                                          # [REQUIRED]
                "link": "https://www.bfe.admin.ch/bfe/en/home/data-and-statistics.html",          # [OPTIONAL]
                "access_date": "2026-06-01",                                                      # [OPTIONAL] ISO 8601
                "assessment_method": "literature",                                                # [OPTIONAL]
                "assessment_notes": "Table 4.2, Page 12"                                          # [OPTIONAL]
            }
        ]
    },
    
    # --- VALUES (List of items with uncertainties) ---
    "values": {
        "_item": [
            {
                "attribute_name": "Capital Expenditure",
                "attribute_description": "Initial investment cost per kW",
                "unit": "CHF/kW",
                "value": 1450.0,
                "value_format": "float",
                
                # Uncertainty Block (Optional)
                "uncertainty": {
                    "uncertainty_type": "range",
                    "upper_value": 1600.0,
                    "lower_value": 1300.0,
                    "other_value": None,
                    "uncertainty_assumption": "Based on 2024 market variance in Zurich."
                },
                
                "scenario": "Base Case",
                "time_index": 2025
            },
            {
                "attribute_name": "System Efficiency",
                "attribute_description": "Overall conversion efficiency under STC",
                "unit": "fraction",
                "value": 0.18,
                "value_format": "float",
                "uncertainty": None,  # No uncertainty provided for this specific value
                "scenario": "Optimistic",
                "time_index": 2025
            }
        ]
    },
    
    # --- TAGS & METADATA ---
    "tags": {
        "related_project": "Swiss Energy Transition Study 2026",  # [OPTIONAL]
        "tags": ["renewable", "solar", "Swiss", "residential", "PV"]  # [OPTIONAL] List of strings
    }
}

# --- HELPER: Print formatted output for verification ---
print("=== Generated Record Preview ===")
pprint(unmapped_record, width=100, indent=1, compact=True, sort_dicts=False)

=== Generated Record Preview ===
{'technology_name': '5 kW Residential Rooftop Solar PV',
 'description': 'A 5 kW rooftop photovoltaic system in Switzerland with 18% efficiency, installed '
                'on a south-facing slope.',
 'technology': {'technology_type': 'conversion',
                'technology_category': 'renewable',
                'technology_assumption': 'Data based on 2024 Swiss Federal Office of Energy (SFOE) '
                                         'demo plant specs.',
                'process_name': 'Electricity Generation',
                'process_type': 'conversion',
                'process_category': 'power',
                'process_assumption': 'Assuming standard irradiance conditions for Zurich region.'},
 'scope': {'geographic_scope_description': 'Switzerland (Zurich Canton)',
           'temporal_scope_description': '2025-2030',
           'capacity_scope_description': 'residential (1-10 kW)',
           'system_boundary_description': 'grid-connected,

## Step 2 - User Input

Load one user input record (YAML). You can switch to another file path later if needed.

In [4]:
with TEST_RECORD_PATH.open("r", encoding="utf-8") as f:
    USER_INPUT = yaml.safe_load(f)

print("Loaded from:", TEST_RECORD_PATH)
print("Loaded record keys:", list(USER_INPUT.keys()))
print("Technology name:", USER_INPUT.get("technology_name"))
print("Number of sources:", len(USER_INPUT.get("sources", {}).get("_item", [])))
print("Number of values:", len(USER_INPUT.get("values", {}).get("_item", [])))

pprint(USER_INPUT)

Loaded from: e:\Barton\repositories\motel-platform\dev2\data\test_records\r00001.yaml
Loaded record keys: ['technology_name', 'description', 'technology', 'scope', 'sources', 'values', 'tags']
Technology name: Autothermal Decentralized Pyrolysis Reactor for Biochar Production
Number of sources: 2
Number of values: 22
{'description': 'A techno-economic analysis of biochar production from '
                'residual forestry biomass (e.g., acacia, reed, broom, gorse) '
                'using an autothermal decentralized pyrolysis reactor in '
                'Central Portugal. The process valorizes low-quality biomass '
                'to prevent wildfires, produce biochar, and generate carbon '
                'credits. Three business scenarios were analyzed, differing in '
                'revenue streams: (1) biochar + carbon credits, (2) forest '
                'management + biochar + carbon credits, and (3) forest '
                'management + biochar + carbon credits + sale of 

## Step 3 - Data Validation

Check required fields and basic structure before mapping.

In [ ]:
def validate_unmapped_record(record: dict) -> list[str]:
    errors = []

    required_top = ["technology_name", "description", "technology", "scope", "sources", "values"]
    for key in required_top:
        if key not in record or record.get(key) in (None, "", {}):
            errors.append(f"Missing required top-level field: {key}")

    source_items = record.get("sources", {}).get("_item", [])
    if not isinstance(source_items, list) or not source_items:
        errors.append("sources._item must be a non-empty list")
    else:
        for i, src in enumerate(source_items, start=1):
            for k in ["source_description", "source_type"]:
                if not src.get(k):
                    errors.append(f"sources._item[{i}] missing {k}")

    value_items = record.get("values", {}).get("_item", [])
    if not isinstance(value_items, list) or not value_items:
        errors.append("values._item must be a non-empty list")
    else:
        for i, val in enumerate(value_items, start=1):
            for k in ["attribute_name", "unit", "value"]:
                if val.get(k) in (None, ""):
                    errors.append(f"values._item[{i}] missing {k}")

    return errors

In [ ]:
VALIDATION_ERRORS = validate_unmapped_record(USER_INPUT)

if VALIDATION_ERRORS:
    print("Validation failed:")
    for err in VALIDATION_ERRORS:
        print(" -", err)
else:
    print("Validation passed: input is ready for mapping.")

Validation passed: input is ready for mapping.


## Step 4 - Data Mapping Suggestion

Suggest links to existing technology, process, and attribute vocabularies using text similarity.

TODO: replace this by LLM:
 - do it in chat!
 - do it with API/packages...
 - do it with local LLM

In [6]:
def load_csv_rows(path: Path) -> list[dict]:
    if not path.exists() or path.stat().st_size == 0:
        return []
    with path.open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "").strip().lower())


def best_match(query: str, rows: list[dict], name_key: str, id_key: str) -> dict | None:
    if not rows:
        return None
    query_norm = normalize(query)
    best = None
    for row in rows:
        candidate = normalize(row.get(name_key, ""))
        score = difflib.SequenceMatcher(None, query_norm, candidate).ratio()
        if best is None or score > best["score"]:
            best = {
                "query": query,
                "candidate_name": row.get(name_key, ""),
                "candidate_id": row.get(id_key, ""),
                "score": round(score, 3),
            }
    return best

In [7]:
tech_rows = load_csv_rows(DB_DIR / "secondary" / "technology.csv")
process_rows = load_csv_rows(DB_DIR / "secondary" / "process.csv")
attribute_rows = load_csv_rows(DB_DIR / "controlled_vocabulary" / "attribute.csv")

SUGGESTIONS = {
    "technology": best_match(USER_INPUT.get("technology_name", ""), tech_rows, "technology_name", "tech_id"),
    "process": best_match(
        USER_INPUT.get("technology", {}).get("process_name", ""),
        process_rows,
        "process_name",
        "process_id",
    ),
    "attributes": [],
}

for item in USER_INPUT.get("values", {}).get("_item", []):
    suggestion = best_match(item.get("attribute_name", ""), attribute_rows, "attribute_name", "attribute_id")
    SUGGESTIONS["attributes"].append(
        {
            "attribute_name": item.get("attribute_name", ""),
            "suggestion": suggestion,
        }
    )

print("Technology suggestion:")
pprint(SUGGESTIONS["technology"])
print("\nProcess suggestion:")
pprint(SUGGESTIONS["process"])
print("\nAttribute suggestions (first 5):")
pprint(SUGGESTIONS["attributes"][:5])

Technology suggestion:
{'candidate_id': 'TECH_0024',
 'candidate_name': 'Solar Thermal Heating',
 'query': 'Autothermal Decentralized Pyrolysis Reactor for Biochar Production',
 'score': 0.322}

Process suggestion:
{'candidate_id': 'PROC_0013',
 'candidate_name': 'Carbon Production',
 'query': 'Pyrolysis',
 'score': 0.308}

Attribute suggestions (first 5):
[{'attribute_name': 'Pyrolysis Reactor Capacity', 'suggestion': None},
 {'attribute_name': 'Biochar Yield', 'suggestion': None},
 {'attribute_name': 'Carbon Sequestration Potential', 'suggestion': None},
 {'attribute_name': 'Net Present Value (NPV) - Scenario 1', 'suggestion': None},
 {'attribute_name': 'Net Present Value (NPV) - Scenario 2', 'suggestion': None}]


## Step 5 - User Confirmation

Auto-accept strong matches, and leave weak matches for manual override.

In [ ]:
AUTO_ACCEPT_THRESHOLD = 0.80

# Optional manual overrides (edit these values when needed)
MANUAL_OVERRIDES = {
    "tech_id": None,
    "process_id": None,
    "attribute_ids": {
        # "Attribute Name": "ATTR_ID"
    },
}

REVIEWED_MAPPING = {
    "tech_id": None,
    "process_id": None,
    "attribute_ids": {},
}

tech_s = SUGGESTIONS.get("technology")
if tech_s and tech_s["score"] >= AUTO_ACCEPT_THRESHOLD:
    REVIEWED_MAPPING["tech_id"] = tech_s["candidate_id"]

proc_s = SUGGESTIONS.get("process")
if proc_s and proc_s["score"] >= AUTO_ACCEPT_THRESHOLD:
    REVIEWED_MAPPING["process_id"] = proc_s["candidate_id"]

for item in SUGGESTIONS.get("attributes", []):
    attr_name = item["attribute_name"]
    sug = item.get("suggestion")
    if sug and sug["score"] >= AUTO_ACCEPT_THRESHOLD:
        REVIEWED_MAPPING["attribute_ids"][attr_name] = sug["candidate_id"]
    else:
        REVIEWED_MAPPING["attribute_ids"][attr_name] = None

# Apply manual overrides when provided
if MANUAL_OVERRIDES.get("tech_id"):
    REVIEWED_MAPPING["tech_id"] = MANUAL_OVERRIDES["tech_id"]
if MANUAL_OVERRIDES.get("process_id"):
    REVIEWED_MAPPING["process_id"] = MANUAL_OVERRIDES["process_id"]
for k, v in MANUAL_OVERRIDES.get("attribute_ids", {}).items():
    REVIEWED_MAPPING["attribute_ids"][k] = v

print("Final reviewed mapping:")
pprint(REVIEWED_MAPPING)

## Step 6 - Add to Database

Write unmapped and mapped records, and add missing linked vocabulary entries when needed.

In [ ]:
def slugify(text: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9]+", "_", (text or "").strip()).strip("_")
    return cleaned.upper()[:40] if cleaned else "UNKNOWN"


def next_numeric_id(prefix: str, existing_ids: list[str], width: int = 5) -> str:
    nums = []
    for x in existing_ids:
        m = re.search(r"(\d+)$", x or "")
        if m:
            nums.append(int(m.group(1)))
    nxt = (max(nums) + 1) if nums else 1
    return f"{prefix}{nxt:0{width}d}"


def ensure_csv_row(path: Path, fieldnames: list[str], key_field: str, row: dict) -> None:
    rows = load_csv_rows(path)
    existing_keys = {r.get(key_field) for r in rows}
    if row.get(key_field) in existing_keys:
        return

    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = (not path.exists()) or path.stat().st_size == 0
    with path.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in fieldnames})

# 1) Ensure linked IDs exist
tech_csv = DB_DIR / "secondary" / "technology.csv"
process_csv = DB_DIR / "secondary" / "process.csv"
attr_csv = DB_DIR / "controlled_vocabulary" / "attribute.csv"
source_csv = DB_DIR / "secondary" / "source.csv"

tech_id = REVIEWED_MAPPING.get("tech_id") or f"T_NEW_{slugify(USER_INPUT.get('technology_name', ''))}"
process_id = REVIEWED_MAPPING.get("process_id") or f"P_NEW_{slugify(USER_INPUT.get('technology', {}).get('process_name', ''))}"

ensure_csv_row(
    process_csv,
    ["process_id", "process_name", "ontology_iri"],
    "process_id",
    {
        "process_id": process_id,
        "process_name": USER_INPUT.get("technology", {}).get("process_name", ""),
        "ontology_iri": "",
    },
)

ensure_csv_row(
    tech_csv,
    ["tech_id", "technology_name", "ontology_iri", "process_id"],
    "tech_id",
    {
        "tech_id": tech_id,
        "technology_name": USER_INPUT.get("technology_name", ""),
        "ontology_iri": "",
        "process_id": process_id,
    },
)

attribute_items = USER_INPUT.get("values", {}).get("_item", [])
resolved_attributes = {}
for item in attribute_items:
    attr_name = item.get("attribute_name", "")
    attr_id = REVIEWED_MAPPING.get("attribute_ids", {}).get(attr_name)
    if not attr_id:
        attr_id = f"ATTR_NEW_{slugify(attr_name)}"

    resolved_attributes[attr_name] = attr_id
    ensure_csv_row(
        attr_csv,
        ["attribute_id", "attribute_name", "attribute_description", "attribute_unit", "ehubX_name", "ontology_iri", "note"],
        "attribute_id",
        {
            "attribute_id": attr_id,
            "attribute_name": attr_name,
            "attribute_description": item.get("attribute_description", ""),
            "attribute_unit": item.get("unit", ""),
            "ehubX_name": "",
            "ontology_iri": "",
            "note": "Added from notebook workflow",
        },
    )

# 2) Add source records if missing
existing_sources = load_csv_rows(source_csv)
existing_source_ids = [r.get("source_id", "") for r in existing_sources]
source_ids = []
for src in USER_INPUT.get("sources", {}).get("_item", []):
    matched = None
    for row in existing_sources:
        if normalize(row.get("source_description", "")) == normalize(src.get("source_description", "")) and normalize(row.get("link", "")) == normalize(src.get("link", "")):
            matched = row
            break

    if matched:
        sid = matched["source_id"]
    else:
        sid = next_numeric_id("SRC_", existing_source_ids, width=5)
        existing_source_ids.append(sid)
        ensure_csv_row(
            source_csv,
            ["source_id", "source_description", "source_type", "link", "access_date", "pdf_backup", "confidence_level", "assessment_method", "assessment_date"],
            "source_id",
            {
                "source_id": sid,
                "source_description": src.get("source_description", ""),
                "source_type": src.get("source_type", ""),
                "link": src.get("link", ""),
                "access_date": src.get("access_date", ""),
                "pdf_backup": "",
                "confidence_level": "medium",
                "assessment_method": src.get("assessment_method", ""),
                "assessment_date": str(date.today()),
            },
        )
    source_ids.append(sid)

# 3) Write unmapped + mapped records
record_root = DB_DIR / "records"
unmapped_dir = record_root / "unmapped_records"
mapped_dir = record_root / "mapped_records"
unmapped_dir.mkdir(parents=True, exist_ok=True)
mapped_dir.mkdir(parents=True, exist_ok=True)

existing_record_ids = [p.stem for p in list(unmapped_dir.glob("R*.yaml")) + list(mapped_dir.glob("R*.yaml"))]
record_id = next_numeric_id("R", existing_record_ids, width=5)

unmapped_path = unmapped_dir / f"{record_id}.yaml"
mapped_path = mapped_dir / f"{record_id}.yaml"

with unmapped_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(USER_INPUT, f, sort_keys=False, allow_unicode=True)

mapped_record = {
    "record_id": record_id,
    "version": {
        "version_number": "1.0",
        "date_created": str(date.today()),
        "date_modified": str(date.today()),
    },
    "tech_id": tech_id,
    "source_id": source_ids[0] if source_ids else "",
    "assumption_id": "",
    "scope": {
        "geographic_scope": USER_INPUT.get("scope", {}).get("geographic_scope_description", ""),
        "temporal_scope": USER_INPUT.get("scope", {}).get("temporal_scope_description", ""),
        "capacity_scope": USER_INPUT.get("scope", {}).get("capacity_scope_description", ""),
        "system_boundary": USER_INPUT.get("scope", {}).get("system_boundary_description", ""),
    },
    "values": {
        "_item": [
            {
                "attribute_id": resolved_attributes.get(v.get("attribute_name", ""), ""),
                "value": v.get("value"),
                "value_format": v.get("value_format", ""),
                "value_type": "numeric" if isinstance(v.get("value"), (int, float)) else "text",
                "unit": v.get("unit", ""),
                "uncertainty": v.get("uncertainty", {}),
                "note": v.get("scenario", ""),
            }
            for v in attribute_items
        ]
    },
}

with mapped_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(mapped_record, f, sort_keys=False, allow_unicode=True)

print("Saved files:")
print(" -", unmapped_path)
print(" -", mapped_path)
print("\nLinked IDs:")
print(" - tech_id:", tech_id)
print(" - process_id:", process_id)
print(" - source_ids:", source_ids[:3], "..." if len(source_ids) > 3 else "")